# Installation

In [ ]:
!pip install --upgrade keras-hub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 33.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.5/62.5 MB 22.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 620.7/620.7 MB 742.9 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.5/57.5 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.5/24.5 MB 107.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 144.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 136.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 226.5/226.5 kB 22.5 MB/s eta 0:00:00


# Environment & Initial imports

In [ ]:
import os, sys, time, textwrap, warnings
warnings.filterwarnings("ignore")

In [ ]:
# ── CRITICAL: set backend BEFORE importing keras ──────────────────────
os.environ["KERAS_BACKEND"] = "jax"
# Give JAX/XLA access to ~90 % of available HBM; leave some for overhead
os.environ["XLA_PYTHON_CLIENT_MEM_FRACTION"] = "0.9"
# Speed up compilation on subsequent calls
os.environ["XLA_FLAGS"] = "--xla_gpu_deterministic_ops=false"

# Next Set of Imports

In [ ]:
import jax
import jax.numpy as jnp
import numpy as np
import keras
import keras_hub
from datasets import load_dataset
import tensorflow as tf

# Helper Function

In [ ]:
DIVIDER  = "═" * 70
DIVIDER2 = "─" * 70

def banner(title: str) -> None:
    print(f"\n{DIVIDER}")
    print(f"  {title}")
    print(DIVIDER)

def section(title: str) -> None:
    print(f"\n{DIVIDER2}")
    print(f"  {title}")
    print(DIVIDER2)

banner("Gemma 3 LoRA Fine-tuning on TPU v5e-1")
print(f"  JAX version   : {jax.__version__}")
print(f"  Keras version : {keras.__version__}")
print(f"  keras-hub ver : {keras_hub.__version__}")
print(f"  Devices       : {jax.devices()}")
print(f"  Device kind   : {jax.devices()[0].device_kind}")

if "tpu" not in jax.devices()[0].device_kind.lower():
    print("\n  ⚠  WARNING: No TPU detected. Make sure you selected a TPU runtime.")
    print("     Runtime → Change runtime type → TPU v5e-1  (or TPU v2)")
else:
    print(f"\n  ✓  TPU confirmed: {len(jax.devices())} core(s) available.")


══════════════════════════════════════════════════════════════════════
  Gemma 3 LoRA Fine-tuning on TPU v5e-1
══════════════════════════════════════════════════════════════════════
  JAX version   : 0.7.2
  Keras version : 3.13.2
  keras-hub ver : 0.28.0
  Devices       : [TpuDevice(id=0, process_index=0, coords=(0,0,0), core_on_chip=0)]
  Device kind   : TPU v5 lite

  ✓  TPU confirmed: 1 core(s) available.


In [ ]:
!tpu-info

Libtpu version: 0.0.21                                                          
Accelerator type: v5e                                                           
                                                                                
TPU Chips                                       
┏━━━━━━━━━━━━━┳━━━━━━━━━━━━━━┳━━━━━━━━━┳━━━━━━━┓
┃ Chip        ┃ Type         ┃ Devices ┃ PID   ┃
┡━━━━━━━━━━━━━╇━━━━━━━━━━━━━━╇━━━━━━━━━╇━━━━━━━┩
│ /dev/vfio/0 │ TPU v5e chip │ 1       │ 12449 │
└─────────────┴──────────────┴─────────┴───────┘
TPU Runtime Utilization                     
┏━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┓
┃ Chip ┃ HBM Usage (GiB)      ┃ Duty cycle ┃
┡━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━┩
│ 0    │ 0.00 GiB / 15.75 GiB │ 0.00%      │
└──────┴──────────────────────┴────────────┘
TensorCore Utilization              
┏━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Core ID ┃ TensorCore Utilization ┃
┡━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━┩
│ 0       │ N/A                    │
└───────

# Hyperparameters & Configs

In [ ]:
banner("Configuration")

MODEL_PRESET = "gemma3_250m_it_en"

# ── LoRA ──────────────────────────────────────────────────────────────
LORA_RANK = 4          # rank of the adapter matrices
# Only ~0.1 % of total params become trainable — very memory efficient

# ── Training ──────────────────────────────────────────────────────────
SEQUENCE_LENGTH = 256   # shorter = faster & less memory; increase if needed
BATCH_SIZE      = 2     # raise to 4 if you have head room after lm.summary()
LEARNING_RATE   = 2e-4  # higher LR works well with LoRA
EPOCHS          = 1     # 1 epoch over 1 000 examples is enough for a clear demo
TRAIN_SIZE      = 5000  # subset of the Alpaca dataset to train on

# ── Generation ────────────────────────────────────────────────────────
MAX_NEW_TOKENS  = 128   # tokens to generate during inference

print(f"  Model          : {MODEL_PRESET}")
print(f"  LoRA rank      : {LORA_RANK}")
print(f"  Seq. length    : {SEQUENCE_LENGTH}")
print(f"  Batch size     : {BATCH_SIZE}")
print(f"  Learning rate  : {LEARNING_RATE}")
print(f"  Epochs         : {EPOCHS}")
print(f"  Training size  : {TRAIN_SIZE}")
print(f"  Max new tokens : {MAX_NEW_TOKENS}")


══════════════════════════════════════════════════════════════════════
  Configuration
══════════════════════════════════════════════════════════════════════
  Model          : gemma3_250m_it_en
  LoRA rank      : 4
  Seq. length    : 256
  Batch size     : 2
  Learning rate  : 0.0002
  Epochs         : 1
  Training size  : 5000
  Max new tokens : 128


# Load & format dataset

In [ ]:
banner("Loading Dataset: tatsu-lab/alpaca")

raw_ds = load_dataset("tatsu-lab/alpaca", split="train")
print(f"  Total examples in dataset : {len(raw_ds)}")
print(f"  Features                  : {list(raw_ds.features.keys())}")

# ── Gemma 3 IT chat template ──────────────────────────────────────────
#
#  <bos>                               (added automatically by tokeniser)
#  <start_of_turn>user
#  {instruction}
#
#  Input:                              (only if non-empty)
#  {input}
#  <end_of_turn>
#  <start_of_turn>model
#  {output}<end_of_turn>
#
PROMPT_TEMPLATE = (
    "<start_of_turn>user\n"
    "{instruction}{input_block}"
    "<end_of_turn>\n"
    "<start_of_turn>model\n"
)

def fmt_prompt(instruction: str, input_text: str = "") -> str:
    """Build the user-turn prompt string (no response — used for both train & infer)."""
    input_block = (
        f"\n\nInput:\n{input_text.strip()}" if input_text.strip() else ""
    )
    return PROMPT_TEMPLATE.format(
        instruction=instruction.strip(),
        input_block=input_block,
    )

def fmt_response(output: str) -> str:
    """Wrap the target output in the model-turn markers."""
    return output.strip() + "<end_of_turn>"

# Alias kept for inference calls in later cells
fmt_infer = fmt_prompt

def extract_response(full_output: str, prompt: str) -> str:
    """Strip the prompt prefix and trailing <end_of_turn> token."""
    response = full_output[len(prompt):]
    if "<end_of_turn>" in response:
        response = response[: response.index("<end_of_turn>")]
    return response.strip()

# ── Sample selection (balanced) ────────────────────────────────────────
with_ctx    = [ex for ex in raw_ds if ex.get("input", "").strip()]
without_ctx = [ex for ex in raw_ds if not ex.get("input", "").strip()]

n_with = min(len(with_ctx), TRAIN_SIZE // 3)
n_wo   = TRAIN_SIZE - n_with

selected = (
    list(with_ctx[:n_with])
    + list(without_ctx[:n_wo])
)

# Gemma3CausalLMPreprocessor requires x = {"prompts": [...], "responses": [...]}
train_prompts   = [fmt_prompt(ex["instruction"], ex.get("input", "")) for ex in selected]
train_responses = [fmt_response(ex["output"])                         for ex in selected]

train_dataset = (
    tf.data.Dataset.from_tensor_slices({
        "prompts":   train_prompts,
        "responses": train_responses,
    })
    .shuffle(buffer_size=len(train_prompts), seed=42)
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)

print(f"\n  Training split:")
print(f"    Examples with context    : {n_with}")
print(f"    Examples without context : {n_wo}")
print(f"    Total training examples  : {len(selected)}")
print(f"\n  Sample prompt (first 400 chars):")
print(f"  {'·' * 60}")
for line in train_prompts[0][:400].split("\n"):
    print(f"    {line}")
print(f"  {'·' * 60}")
print(f"\n  Sample response (first 200 chars):")
print(f"    {train_responses[0][:200]}")


══════════════════════════════════════════════════════════════════════
  Loading Dataset: tatsu-lab/alpaca
══════════════════════════════════════════════════════════════════════


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001-a09b74b3ef9c3b(…):   0%|          | 0.00/24.2M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/52002 [00:00<?, ? examples/s]

  Total examples in dataset : 52002
  Features                  : ['instruction', 'input', 'output', 'text']

  Training split:
    Examples with context    : 1666
    Examples without context : 3334
    Total training examples  : 5000

  Sample prompt (first 400 chars):
  ····························································
    <start_of_turn>user
    Identify the odd one out.
    
    Input:
    Twitter, Instagram, Telegram<end_of_turn>
    <start_of_turn>model
    
  ····························································

  Sample response (first 200 chars):
    Telegram<end_of_turn>


# Define test prompts for before/after comparison

In [ ]:
banner("Test Prompts (used for before & after comparison)")

# These five tasks span different Alpaca skills — classification,
# creative writing, explanation, translation, and list generation.
TEST_PROMPTS = [
    {
        "label"      : "Sentiment Classification",
        "instruction": (
            "Classify the following sentence as Positive, Negative, "
            "or Neutral, and briefly explain your reasoning."
        ),
        "input"      : (
            "The restaurant had wonderful food but the service was painfully slow "
            "and the staff seemed completely indifferent to the customers."
        ),
    },
    {
        "label"      : "Creative Writing",
        "instruction": "Write a short haiku about artificial intelligence.",
        "input"      : "",
    },
    {
        "label"      : "Technical Explanation",
        "instruction": (
            "Explain the difference between RAM and ROM to a 10-year-old "
            "using a simple real-world analogy."
        ),
        "input"      : "",
    },
    {
        "label"      : "Information Extraction",
        "instruction": (
            "Extract all named entities (people, organisations, locations) "
            "from the following text and list them by category."
        ),
        "input"      : (
            "Elon Musk announced that Tesla will open a new Gigafactory in "
            "Austin, Texas, in partnership with Panasonic and the local "
            "government of Travis County."
        ),
    },
    {
        "label"      : "Actionable List",
        "instruction": "Give five concrete tips for improving sleep quality.",
        "input"      : "",
    },
]

for i, tp in enumerate(TEST_PROMPTS, 1):
    print(f"\n  [{i}] {tp['label']}")
    print(f"      Instruction: {tp['instruction'][:80]}{'...' if len(tp['instruction'])>80 else ''}")
    if tp["input"]:
        print(f"      Input      : {tp['input'][:80]}{'...' if len(tp['input'])>80 else ''}")



══════════════════════════════════════════════════════════════════════
  Test Prompts (used for before & after comparison)
══════════════════════════════════════════════════════════════════════

  [1] Sentiment Classification
      Instruction: Classify the following sentence as Positive, Negative, or Neutral, and briefly e...
      Input      : The restaurant had wonderful food but the service was painfully slow and the sta...

  [2] Creative Writing
      Instruction: Write a short haiku about artificial intelligence.

  [3] Technical Explanation
      Instruction: Explain the difference between RAM and ROM to a 10-year-old using a simple real-...

  [4] Information Extraction
      Instruction: Extract all named entities (people, organisations, locations) from the following...
      Input      : Elon Musk announced that Tesla will open a new Gigafactory in Austin, Texas, in ...

  [5] Actionable List
      Instruction: Give five concrete tips for improving sleep quality.


# Load Gemma 3 Instruct Model

In [ ]:
banner(f"Loading model: {MODEL_PRESET}")
print("  (First run: downloads ~540 MB — may take a few minutes)")
print("  (Subsequent runs: loads from cache)")

t0 = time.time()

gemma_lm = keras_hub.models.Gemma3CausalLM.from_preset(
    "hf://google/gemma-3-270m-it"
)

# Cap the sequence length to our configured value
gemma_lm.preprocessor.sequence_length = SEQUENCE_LENGTH

elapsed = time.time() - t0
print(f"\n  ✓ Model loaded in {elapsed:.1f}s")

# Parameter summary
total_params = sum(np.prod(w.shape) for w in gemma_lm.weights)
print(f"  Total parameters : {total_params:,}")
print(f"  Approx. size (bf16): {total_params * 2 / 1e9:.2f} GB")



══════════════════════════════════════════════════════════════════════
  Loading model: gemma3_250m_it_en
══════════════════════════════════════════════════════════════════════
  (First run: downloads ~540 MB — may take a few minutes)
  (Subsequent runs: loads from cache)


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/536M [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/4.69M [00:00<?, ?B/s]


  ✓ Model loaded in 9.6s
  Total parameters : 268,098,176
  Approx. size (bf16): 0.54 GB


# Inference BEFORE fine-tuning

In [ ]:
banner("INFERENCE — BEFORE Fine-tuning")
print("  Running inference on all test prompts with the BASE model...\n")

before_results: dict[str, str] = {}

for i, tp in enumerate(TEST_PROMPTS, 1):
    prompt   = fmt_infer(tp["instruction"], tp["input"])
    max_len  = len(prompt.split()) + MAX_NEW_TOKENS   # rough token budget

    section(f"[{i}/{len(TEST_PROMPTS)}]  {tp['label']}")
    print(f"  Instruction : {tp['instruction']}")
    if tp["input"]:
        print(f"  Input       : {tp['input']}")

    t0  = time.time()
    out = gemma_lm.generate(prompt, max_length=SEQUENCE_LENGTH)
    resp = extract_response(out, prompt)
    elapsed = time.time() - t0

    print(f"\n  ◀ Response BEFORE fine-tuning  ({elapsed:.1f}s):")
    for line in textwrap.wrap(resp or "(empty response)", width=66):
        print(f"    {line}")

    before_results[tp["label"]] = resp

print(f"\n  ✓ Pre-training inference done for all {len(TEST_PROMPTS)} prompts.")



══════════════════════════════════════════════════════════════════════
  INFERENCE — BEFORE Fine-tuning
══════════════════════════════════════════════════════════════════════
  Running inference on all test prompts with the BASE model...


──────────────────────────────────────────────────────────────────────
  [1/5]  Sentiment Classification
──────────────────────────────────────────────────────────────────────
  Instruction : Classify the following sentence as Positive, Negative, or Neutral, and briefly explain your reasoning.
  Input       : The restaurant had wonderful food but the service was painfully slow and the staff seemed completely indifferent to the customers.

  ◀ Response BEFORE fine-tuning  (51.7s):
    **Positive**

──────────────────────────────────────────────────────────────────────
  [2/5]  Creative Writing
──────────────────────────────────────────────────────────────────────
  Instruction : Write a short haiku about artificial intelligence.

  ◀ Response BEFORE 

# LORA Finetuning

In [ ]:
banner(f"Enabling LoRA (rank = {LORA_RANK})")

# This replaces the dense projection matrices in every attention layer
# with low-rank adapter pairs (A, B).  Frozen weights are kept in full
# precision; only the tiny adapter matrices are trainable.
gemma_lm.backbone.enable_lora(rank=LORA_RANK)

trainable = sum(np.prod(w.shape) for w in gemma_lm.trainable_weights)
total     = sum(np.prod(w.shape) for w in gemma_lm.weights)

print(f"  Trainable parameters : {trainable:,}")
print(f"  Frozen  parameters   : {total - trainable:,}")
print(f"  Total   parameters   : {total:,}")
print(f"  Trainable ratio      : {100 * trainable / total:.3f}%")
print(f"\n  LoRA adds only {trainable / 1e6:.2f}M trainable parameters — very efficient!")



══════════════════════════════════════════════════════════════════════
  Enabling LoRA (rank = 4)
══════════════════════════════════════════════════════════════════════
  Trainable parameters : 267,264
  Frozen  parameters   : 268,098,176
  Total   parameters   : 268,365,440
  Trainable ratio      : 0.100%

  LoRA adds only 0.27M trainable parameters — very efficient!


# Compile & train

In [ ]:
banner("Compiling & Training")

# bfloat16 keeps memory usage half of float32 with near-identical quality
# (TPU natively accelerates bfloat16 arithmetic)
keras.mixed_precision.set_global_policy("bfloat16")

gemma_lm.compile(
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    optimizer=keras.optimizers.AdamW(
        learning_rate=LEARNING_RATE,
        weight_decay=0.01,
    ),
    weighted_metrics=[keras.metrics.SparseCategoricalAccuracy()],
    # jit_compile=True   # uncomment if you want XLA-compiled training step
)

steps_per_epoch = len(selected) // BATCH_SIZE
print(f"  Training examples  : {len(selected)}")
print(f"  Batch size         : {BATCH_SIZE}")
print(f"  Steps / epoch      : {steps_per_epoch}")
print(f"  Epochs             : {EPOCHS}")
print(f"  Effective LR       : {LEARNING_RATE}")
print(f"\n  Starting training...")
print(f"  (First step triggers XLA compilation — may take ~2 min, then speeds up)\n")

t0      = time.time()
history = gemma_lm.fit(
    train_dataset,
    epochs=3,
)
elapsed = time.time() - t0

final_loss = history.history["loss"][-1]
final_acc  = history.history.get("sparse_categorical_accuracy", [None])[-1]

print(f"\n  ✓ Training complete in {elapsed/60:.1f} min  ({elapsed:.0f}s)")
print(f"  Final loss     : {final_loss:.4f}")
if final_acc is not None:
    print(f"  Final accuracy : {final_acc:.4f}")

# ── Training loss curve (text sparkline) ──────────────────────────────
losses = history.history["loss"]
if len(losses) > 1:
    lo, hi = min(losses), max(losses)
    blocks = " ▁▂▃▄▅▆▇█"
    spark  = "".join(
        blocks[int((v - lo) / (hi - lo + 1e-9) * 8)] for v in losses
    )
    print(f"\n  Loss curve : {spark}  ({hi:.3f} → {lo:.3f})")



══════════════════════════════════════════════════════════════════════
  Compiling & Training
══════════════════════════════════════════════════════════════════════
  Training examples  : 5000
  Batch size         : 2
  Steps / epoch      : 2500
  Epochs             : 1
  Effective LR       : 0.0002

  Starting training...
  (First step triggers XLA compilation — may take ~2 min, then speeds up)

Epoch 1/3
2500/2500 ━━━━━━━━━━━━━━━━━━━━ 73s 8ms/step - loss: 0.3954 - sparse_categorical_accuracy: 0.5861
Epoch 2/3
2500/2500 ━━━━━━━━━━━━━━━━━━━━ 21s 8ms/step - loss: 0.3808 - sparse_categorical_accuracy: 0.5951
Epoch 3/3
2500/2500 ━━━━━━━━━━━━━━━━━━━━ 21s 8ms/step - loss: 0.3720 - sparse_categorical_accuracy: 0.6021

  ✓ Training complete in 2.0 min  (117s)
  Final loss     : 0.3720
  Final accuracy : 0.6021

  Loss curve : ▇▂   (0.395 → 0.372)


# Inference AFTER fine-tuning

In [ ]:
banner("INFERENCE — AFTER Fine-tuning")
print("  Running inference on all test prompts with the LoRA-adapted model...\n")

after_results: dict[str, str] = {}

for i, tp in enumerate(TEST_PROMPTS, 1):
    prompt  = fmt_infer(tp["instruction"], tp["input"])

    section(f"[{i}/{len(TEST_PROMPTS)}]  {tp['label']}")
    print(f"  Instruction : {tp['instruction']}")
    if tp["input"]:
        print(f"  Input       : {tp['input']}")

    t0   = time.time()
    out  = gemma_lm.generate(prompt, max_length=SEQUENCE_LENGTH)
    resp = extract_response(out, prompt)
    elapsed = time.time() - t0

    print(f"\n  ▶ Response AFTER fine-tuning  ({elapsed:.1f}s):")
    for line in textwrap.wrap(resp or "(empty response)", width=66):
        print(f"    {line}")

    after_results[tp["label"]] = resp

print(f"\n  ✓ Post-training inference done for all {len(TEST_PROMPTS)} prompts.")



══════════════════════════════════════════════════════════════════════
  INFERENCE — AFTER Fine-tuning
══════════════════════════════════════════════════════════════════════
  Running inference on all test prompts with the LoRA-adapted model...


──────────────────────────────────────────────────────────────────────
  [1/5]  Sentiment Classification
──────────────────────────────────────────────────────────────────────
  Instruction : Classify the following sentence as Positive, Negative, or Neutral, and briefly explain your reasoning.
  Input       : The restaurant had wonderful food but the service was painfully slow and the staff seemed completely indifferent to the customers.

  ▶ Response AFTER fine-tuning  (42.2s):
    The sentence is classified as Positive. The sentence expresses a
    positive sentiment about the restaurant's food and service. The
    sentence also describes a negative situation with a negative
    outcome.

────────────────────────────────────────────────────

# Side-by-side comparison

In [ ]:
banner("FULL COMPARISON: BEFORE  vs  AFTER  LoRA Fine-tuning")

for i, tp in enumerate(TEST_PROMPTS, 1):
    label = tp["label"]
    print(f"\n{'━' * 70}")
    print(f"  Task [{i}]: {label}")
    print(f"  Instruction: {tp['instruction']}")
    if tp["input"]:
        print(f"  Input      : {tp['input']}")

    print(f"\n  ◀ BEFORE fine-tuning:")
    b_text = before_results.get(label, "(no result)")
    for line in textwrap.wrap(b_text, width=64):
        print(f"    {line}")

    print(f"\n  ▶ AFTER  fine-tuning (LoRA rank={LORA_RANK}, {TRAIN_SIZE} examples):")
    a_text = after_results.get(label, "(no result)")
    for line in textwrap.wrap(a_text, width=64):
        print(f"    {line}")

print(f"\n{'━' * 70}")
print(f"""
  Summary
  ───────
  Model     : {MODEL_PRESET}
  LoRA rank : {LORA_RANK}  →  {trainable/1e6:.2f}M trainable params  ({100*trainable/total:.3f}% of total)
  Data      : {len(selected)} Alpaca examples  ×  {EPOCHS} epoch(s)
  Final loss: {final_loss:.4f}
  Runtime   : TPU v5e-1 (JAX backend, bfloat16 mixed precision)
""")


══════════════════════════════════════════════════════════════════════
  FULL COMPARISON: BEFORE  vs  AFTER  LoRA Fine-tuning
══════════════════════════════════════════════════════════════════════

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  Task [1]: Sentiment Classification
  Instruction: Classify the following sentence as Positive, Negative, or Neutral, and briefly explain your reasoning.
  Input      : The restaurant had wonderful food but the service was painfully slow and the staff seemed completely indifferent to the customers.

  ◀ BEFORE fine-tuning:
    **Positive**

  ▶ AFTER  fine-tuning (LoRA rank=4, 5000 examples):
    The sentence is classified as Positive. The sentence expresses a
    positive sentiment about the restaurant's food and service. The
    sentence also describes a negative situation with a negative
    outcome.

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  Task [2]: Creative Writing
  Instruction: Wr

# Save & reload LoRA weights

In [ ]:
banner("Saving LoRA Adapter Weights")

SAVE_PATH = "gemma3_1b_lora_alpaca.weights.h5"
gemma_lm.save_weights(SAVE_PATH)
print(f"  ✓ Adapter weights saved to: {SAVE_PATH}")
print(f"""
  To reload later in a fresh session:
  ────────────────────────────────────
      import os, keras, keras_hub
      os.environ["KERAS_BACKEND"] = "jax"

      gemma_lm = keras_hub.models.GemmaCausalLM.from_preset("{MODEL_PRESET}")
      gemma_lm.backbone.enable_lora(rank={LORA_RANK})
      gemma_lm.load_weights("{SAVE_PATH}")
      # model is now fine-tuned!
""")


══════════════════════════════════════════════════════════════════════
  Saving LoRA Adapter Weights
══════════════════════════════════════════════════════════════════════
  ✓ Adapter weights saved to: gemma3_1b_lora_alpaca.weights.h5

  To reload later in a fresh session:
  ────────────────────────────────────
      import os, keras, keras_hub
      os.environ["KERAS_BACKEND"] = "jax"

      gemma_lm = keras_hub.models.GemmaCausalLM.from_preset("gemma3_250m_it_en")
      gemma_lm.backbone.enable_lora(rank=4)
      gemma_lm.load_weights("gemma3_1b_lora_alpaca.weights.h5")
      # model is now fine-tuned!

